# SAR → Optical Terrain Classifier
### ResNet-18 Fine-tuned on Real Optical Images → Inference on CycleGAN Generated Images
**Pipeline:**
1. Train ResNet-18 on real Sentinel-2 optical images (4 terrain classes)
2. Generate fake optical images from SAR using trained G_AB (CycleGAN)
3. Classify generated images → compare accuracy vs real optical baseline


In [ ]:
# All required libraries are pre-installed on Kaggle
# Just verify versions
import torch, torchvision
print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
print(f'GPU count   : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}     : {torch.cuda.get_device_name(i)}')


In [ ]:
import os, random, time, json, zipfile, glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
class Config:
    # ── PATHS ──────────────────────────────────────────────────────────────
    # Same dataset as CycleGAN notebook
    DATA_ROOT      = '/kaggle/input/sentinel12-image-pairs-segregated-by-terrain/v_2'
    WORKING_ROOT   = '/kaggle/working'
    CHECKPOINT_DIR = '/kaggle/working/checkpoints'
    OUTPUT_DIR     = '/kaggle/working/outputs'

    # ── Path to your CycleGAN best_model.pth ───────────────────────────────
    # Upload best_model.pth as a Kaggle dataset, then update this path
    # Or if running in same session: '/kaggle/working/checkpoints/best_model.pth'
    CYCLEGAN_CKPT  = '/kaggle/input/sarnet-checkpoints/best_model.pth'

    # ── DATASET ────────────────────────────────────────────────────────────
    CLASSES        = ['agri', 'barrenland', 'grassland', 'urban']
    NUM_CLASSES    = 4
    # Samples per class for classifier training on real optical images
    MAX_PER_CLASS  = 500   # 500×4 = 2000 total — matches CycleGAN dataset size

    # ── IMAGE ──────────────────────────────────────────────────────────────
    IMG_SIZE       = 256
    SAR_CHANNELS   = 1
    OPT_CHANNELS   = 3

    # ── CycleGAN Generator params (must match your trained model) ──────────
    N_RES_BLOCKS   = 6
    NGF            = 64

    # ── CLASSIFIER TRAINING ────────────────────────────────────────────────
    # Phase 1: freeze backbone, train FC only
    PHASE1_EPOCHS  = 5
    PHASE1_LR      = 1e-3

    # Phase 2: unfreeze layer3+layer4, fine-tune
    PHASE2_EPOCHS  = 15
    PHASE2_LR      = 1e-4

    BATCH_SIZE     = 32    # classifier is lightweight, larger batch is fine
    WEIGHT_DECAY   = 1e-4
    PATIENCE       = 7     # early stopping for classifier

    # ── INFERENCE ──────────────────────────────────────────────────────────
    # How many SAR val images to generate and classify per class
    INFER_PER_CLASS = 100  # 100×4 = 400 generated images total

    USE_AMP        = True


cfg = Config()
os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.OUTPUT_DIR,     exist_ok=True)

print('Config ready.')
print(f'  Classes        : {cfg.CLASSES}')
print(f'  Train per class: {cfg.MAX_PER_CLASS}')
print(f'  Total train    : {cfg.MAX_PER_CLASS * cfg.NUM_CLASSES}')
print(f'  Batch size     : {cfg.BATCH_SIZE}')
print(f'  Phase 1        : {cfg.PHASE1_EPOCHS} epochs (FC only, LR={cfg.PHASE1_LR})')
print(f'  Phase 2        : {cfg.PHASE2_EPOCHS} epochs (fine-tune, LR={cfg.PHASE2_LR})')


In [ ]:
# ── SCAN DATASET ───────────────────────────────────────────────────────────
print('Scanning dataset...')
assert os.path.exists(cfg.DATA_ROOT), f'Dataset not found at {cfg.DATA_ROOT}'

class_image_map = {}   # class_name → list of s2 (optical) image paths
sar_image_map   = {}   # class_name → list of s1 (SAR) image paths

for cls in cfg.CLASSES:
    cls_dir = os.path.join(cfg.DATA_ROOT, cls)
    assert os.path.exists(cls_dir), f'Class folder missing: {cls_dir}'

    # Find s2 (optical) subfolder
    s2_dirs = [d for d in os.listdir(cls_dir) if 's2' in d.lower()]
    s1_dirs = [d for d in os.listdir(cls_dir) if 's1' in d.lower()]
    assert s2_dirs, f'No s2 folder found in {cls_dir}'
    assert s1_dirs, f'No s1 folder found in {cls_dir}'

    s2_dir = os.path.join(cls_dir, s2_dirs[0])
    s1_dir = os.path.join(cls_dir, s1_dirs[0])

    opt_paths = sorted([
        os.path.join(s2_dir, f) for f in os.listdir(s2_dir)
        if f.lower().endswith(('.png', '.jpg', '.tif', '.tiff'))
    ])
    sar_paths = sorted([
        os.path.join(s1_dir, f) for f in os.listdir(s1_dir)
        if f.lower().endswith(('.png', '.jpg', '.tif', '.tiff'))
    ])

    class_image_map[cls] = opt_paths
    sar_image_map[cls]   = sar_paths
    print(f'  {cls:12s} → {len(opt_paths):4d} optical  |  {len(sar_paths):4d} SAR')

print('Dataset scan complete.')


In [ ]:
# ── CLASSIFIER DATASET ─────────────────────────────────────────────────────
class OpticalClassifierDataset(Dataset):
    """
    Loads real Sentinel-2 optical images with terrain class labels.
    Used to train and validate the ResNet-18 classifier.
    """
    def __init__(self, samples, transform):
        # samples: list of (image_path, label_int)
        self.samples   = samples
        self.transform = transform

    def _load(self, path):
        img = Image.open(path)
        return img.convert('RGB')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self._load(path)
        return self.transform(img), label


# ── TRANSFORMS ─────────────────────────────────────────────────────────────
# ImageNet normalization — ResNet-18 pretrained weights expect this
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE + 32, cfg.IMG_SIZE + 32)),
    transforms.RandomCrop(cfg.IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# ── BUILD TRAIN / VAL SPLITS ────────────────────────────────────────────────
train_samples, val_samples = [], []

for label_idx, cls in enumerate(cfg.CLASSES):
    paths = class_image_map[cls].copy()
    random.shuffle(paths)
    paths = paths[:cfg.MAX_PER_CLASS]   # cap at MAX_PER_CLASS

    split = int(0.8 * len(paths))
    for p in paths[:split]:
        train_samples.append((p, label_idx))
    for p in paths[split:]:
        val_samples.append((p, label_idx))

random.shuffle(train_samples)
random.shuffle(val_samples)

train_dataset = OpticalClassifierDataset(train_samples, train_transform)
val_dataset   = OpticalClassifierDataset(val_samples,   val_transform)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f'Train samples : {len(train_samples)}  ({len(train_loader)} batches)')
print(f'Val samples   : {len(val_samples)}  ({len(val_loader)} batches)')
print()
for i, cls in enumerate(cfg.CLASSES):
    tr = sum(1 for _, l in train_samples if l == i)
    va = sum(1 for _, l in val_samples   if l == i)
    print(f'  {cls:12s} → train: {tr}  val: {va}')


In [ ]:
# ── RESNET-18 CLASSIFIER ───────────────────────────────────────────────────
def build_classifier(num_classes=4, pretrained=True):
    """
    ResNet-18 pretrained on ImageNet.
    Final FC layer replaced with Dropout + Linear(512 → num_classes).
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)

    # Replace final FC layer
    in_features = model.fc.in_features   # 512 for ResNet-18
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes)
    )
    return model


classifier = build_classifier(num_classes=cfg.NUM_CLASSES, pretrained=True)

# DataParallel for 2x T4
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs with DataParallel')
    classifier = nn.DataParallel(classifier)

classifier = classifier.to(device)

def count_params(m):
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_params(classifier)
print(f'Total params    : {total/1e6:.2f}M')
print(f'Trainable params: {trainable/1e6:.2f}M')


In [ ]:
# ── FREEZE / UNFREEZE HELPERS ──────────────────────────────────────────────
def get_base(model):
    """Unwrap DataParallel to access ResNet layers directly."""
    return model.module if hasattr(model, 'module') else model

def freeze_backbone(model):
    """Phase 1: freeze all layers except FC head."""
    base = get_base(model)
    for name, param in base.named_parameters():
        if 'fc' not in name:
            param.requires_grad = False
    _, trainable = count_params(model)
    print(f'Backbone FROZEN   — trainable params: {trainable/1e3:.1f}K')

def unfreeze_top_layers(model):
    """Phase 2: unfreeze layer3, layer4, and FC for fine-tuning."""
    base = get_base(model)
    for name, param in base.named_parameters():
        if any(x in name for x in ['layer3', 'layer4', 'fc']):
            param.requires_grad = True
        else:
            param.requires_grad = False
    _, trainable = count_params(model)
    print(f'Top layers UNFROZEN — trainable params: {trainable/1e6:.2f}M')


In [ ]:
# ── TRAIN ONE EPOCH ────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler(enabled=cfg.USE_AMP)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=cfg.USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


def eval_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=cfg.USE_AMP):
                logits = model(imgs)
                loss   = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels


print('Train/eval functions ready.')


In [ ]:
# ── PHASE 1: TRAIN FC HEAD ONLY (backbone frozen) ──────────────────────────
print('='*60)
print('PHASE 1 — FC head only (backbone frozen)')
print('='*60)

freeze_backbone(classifier)

optimizer_p1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, classifier.parameters()),
    lr=cfg.PHASE1_LR, weight_decay=cfg.WEIGHT_DECAY
)

history_clf = defaultdict(list)
best_val_acc = 0.0
patience_ctr = 0

for epoch in range(1, cfg.PHASE1_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc         = train_epoch(classifier, train_loader, optimizer_p1)
    vl_loss, vl_acc, _, _   = eval_epoch(classifier, val_loader)

    history_clf['tr_loss'].append(tr_loss)
    history_clf['tr_acc'].append(tr_acc)
    history_clf['vl_loss'].append(vl_loss)
    history_clf['vl_acc'].append(vl_acc)

    marker = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(get_base(classifier).state_dict(),
                   os.path.join(cfg.CHECKPOINT_DIR, 'best_classifier.pth'))
        marker = ' ★ BEST'

    print(f'  Epoch [{epoch}/{cfg.PHASE1_EPOCHS}] '
          f'tr_loss:{tr_loss:.4f} tr_acc:{tr_acc:.4f} '
          f'vl_loss:{vl_loss:.4f} vl_acc:{vl_acc:.4f} '
          f'[{time.time()-t0:.0f}s]{marker}')

print(f'\nPhase 1 done. Best val acc: {best_val_acc:.4f}')


In [ ]:
# ── PHASE 2: FINE-TUNE LAYER3 + LAYER4 + FC ────────────────────────────────
print('='*60)
print('PHASE 2 — Fine-tune layer3 + layer4 + FC')
print('='*60)

unfreeze_top_layers(classifier)

optimizer_p2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, classifier.parameters()),
    lr=cfg.PHASE2_LR, weight_decay=cfg.WEIGHT_DECAY
)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=cfg.PHASE2_EPOCHS, eta_min=1e-6
)

patience_ctr = 0

for epoch in range(1, cfg.PHASE2_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc         = train_epoch(classifier, train_loader, optimizer_p2)
    vl_loss, vl_acc, _, _   = eval_epoch(classifier, val_loader)
    scheduler_p2.step()

    history_clf['tr_loss'].append(tr_loss)
    history_clf['tr_acc'].append(tr_acc)
    history_clf['vl_loss'].append(vl_loss)
    history_clf['vl_acc'].append(vl_acc)

    marker = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        patience_ctr = 0
        torch.save(get_base(classifier).state_dict(),
                   os.path.join(cfg.CHECKPOINT_DIR, 'best_classifier.pth'))
        marker = ' ★ BEST'
    else:
        patience_ctr += 1

    lr_now = scheduler_p2.get_last_lr()[0]
    print(f'  Epoch [{epoch}/{cfg.PHASE2_EPOCHS}] '
          f'tr_loss:{tr_loss:.4f} tr_acc:{tr_acc:.4f} '
          f'vl_loss:{vl_loss:.4f} vl_acc:{vl_acc:.4f} '
          f'lr:{lr_now:.6f} pat:{patience_ctr}/{cfg.PATIENCE} '
          f'[{time.time()-t0:.0f}s]{marker}')

    if patience_ctr >= cfg.PATIENCE:
        print(f'  Early stopping at epoch {epoch}')
        break

print(f'\nPhase 2 done. Best val acc: {best_val_acc:.4f}')

# Save zip to working root for download
zip_path = os.path.join(cfg.WORKING_ROOT, 'best_classifier.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(os.path.join(cfg.CHECKPOINT_DIR, 'best_classifier.pth'), 'best_classifier.pth')
print(f'Saved: best_classifier.zip ({os.path.getsize(zip_path)/1e6:.1f} MB) — download from Output tab')


In [ ]:
# ── PLOT TRAINING CURVES ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history_clf['tr_loss']) + 1)

axes[0].plot(epochs_range, history_clf['tr_loss'], label='Train Loss', color='royalblue')
axes[0].plot(epochs_range, history_clf['vl_loss'], label='Val Loss',   color='tomato')
axes[0].axvline(cfg.PHASE1_EPOCHS, color='gray', linestyle='--', label='Phase 1→2')
axes[0].set_title('Classifier Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [a*100 for a in history_clf['tr_acc']], label='Train Acc', color='royalblue')
axes[1].plot(epochs_range, [a*100 for a in history_clf['vl_acc']], label='Val Acc',   color='tomato')
axes[1].axvline(cfg.PHASE1_EPOCHS, color='gray', linestyle='--', label='Phase 1→2')
axes[1].set_title('Classifier Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('ResNet-18 Classifier Training — Real Optical Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'classifier_training_curves.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('Training curves saved.')


In [ ]:
# ── EVALUATE ON REAL OPTICAL VAL SET ───────────────────────────────────────
print('Evaluating classifier on REAL optical val set...')

# Load best weights
best_state = torch.load(os.path.join(cfg.CHECKPOINT_DIR, 'best_classifier.pth'),
                        map_location=device)
get_base(classifier).load_state_dict(best_state)

_, real_acc, real_preds, real_labels = eval_epoch(classifier, val_loader)
print(f'Real Optical Val Accuracy: {real_acc*100:.2f}%')
print()
print(classification_report(real_labels, real_preds,
                             target_names=cfg.CLASSES, digits=4))

# Confusion matrix
cm_real = confusion_matrix(real_labels, real_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_real, annot=True, fmt='d', cmap='Blues',
            xticklabels=cfg.CLASSES, yticklabels=cfg.CLASSES, ax=ax)
ax.set_title(f'Real Optical — Confusion Matrix (Acc: {real_acc*100:.1f}%)', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'confusion_matrix_real.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('Confusion matrix (real) saved.')


In [ ]:
# ── CYCLEGAN GENERATOR (must match your trained model exactly) ─────────────
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
        )
    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(self, in_channels, out_channels, ngf=64, n_res_blocks=6):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, ngf, 7, bias=False),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True),

            nn.Conv2d(ngf,   ngf*2, 3, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(ngf*2),
            nn.ReLU(inplace=True),

            nn.Conv2d(ngf*2, ngf*4, 3, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(ngf*4),
            nn.ReLU(inplace=True),
        ]
        for _ in range(n_res_blocks):
            layers.append(ResidualBlock(ngf*4))
        layers += [
            nn.ConvTranspose2d(ngf*4, ngf*2, 3, stride=2, padding=1, output_padding=1, bias=False),
            nn.InstanceNorm2d(ngf*2),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(ngf*2, ngf, 3, stride=2, padding=1, output_padding=1, bias=False),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True),

            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, out_channels, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


print('Generator architecture defined.')


In [ ]:
# ── LOAD TRAINED CYCLEGAN G_AB ─────────────────────────────────────────────
# ⚠️  IMPORTANT: Upload your best_model.pth as a Kaggle dataset first
# Kaggle → New Dataset → upload best_model.pth → name it 'sarnet-checkpoints'
# Then the path cfg.CYCLEGAN_CKPT will work

G_AB = Generator(in_channels=cfg.SAR_CHANNELS,
                 out_channels=cfg.OPT_CHANNELS,
                 ngf=cfg.NGF,
                 n_res_blocks=cfg.N_RES_BLOCKS).to(device)

if os.path.exists(cfg.CYCLEGAN_CKPT):
    ckpt = torch.load(cfg.CYCLEGAN_CKPT, map_location=device)

    # Handle both DataParallel and non-DataParallel saved checkpoints
    state = ckpt.get('G_AB', ckpt)
    # Strip 'module.' prefix if saved with DataParallel
    state = {k.replace('module.', ''): v for k, v in state.items()}
    G_AB.load_state_dict(state)
    G_AB.eval()

    trained_epoch = ckpt.get('epoch', 'unknown')
    print(f'✅ G_AB loaded from epoch {trained_epoch}')
    print(f'   Path: {cfg.CYCLEGAN_CKPT}')
else:
    print(f'❌ CycleGAN checkpoint not found at: {cfg.CYCLEGAN_CKPT}')
    print('   Steps to fix:')
    print('   1. Download best_model.pth from your CycleGAN notebook Output tab')
    print('   2. Go to kaggle.com → Datasets → New Dataset')
    print('   3. Upload best_model.pth → name it sarnet-checkpoints')
    print('   4. Add this dataset to your notebook via + Add Data')
    raise FileNotFoundError('CycleGAN checkpoint required to proceed.')


In [ ]:
# ── SAR DATASET FOR GENERATION ─────────────────────────────────────────────
class SARInferenceDataset(Dataset):
    """Loads SAR images for feeding into G_AB to generate optical images."""
    def __init__(self, sar_paths, img_size=256):
        self.sar_paths = sar_paths
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def _load_sar(self, path):
        img = Image.open(path)
        arr = np.array(img)
        if arr.ndim == 3:
            arr = arr[:, :, 0]
        p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
        arr = np.clip(arr, p2, p98)
        if p98 > p2:
            arr = ((arr - p2) / (p98 - p2) * 255).astype(np.uint8)
        else:
            arr = np.zeros_like(arr, dtype=np.uint8)
        return Image.fromarray(arr, mode='L')

    def __len__(self):
        return len(self.sar_paths)

    def __getitem__(self, idx):
        return self.transform(self._load_sar(self.sar_paths[idx]))


print('SAR inference dataset class defined.')


In [ ]:
# ── GENERATE OPTICAL IMAGES AND CLASSIFY ───────────────────────────────────
print('Generating optical images from SAR and classifying...')
print('='*60)

# Classifier transform for generated images
# Generated images are in [-1, 1] (Tanh output) → convert to ImageNet range
gen_to_classifier = transforms.Compose([
    transforms.Normalize(
        mean=[(0 - m) / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
        std=[1.0 / s for s in IMAGENET_STD]
    )
])

# More direct: denormalize from [-1,1] then re-normalize to ImageNet
def prepare_for_classifier(gen_tensor):
    """
    gen_tensor: (B, 3, H, W) in [-1, 1] from CycleGAN Tanh output
    Returns:    (B, 3, H, W) normalized for ResNet-18 (ImageNet stats)
    """
    # [-1, 1] → [0, 1]
    x = (gen_tensor + 1.0) / 2.0
    x = x.clamp(0, 1)
    # [0, 1] → ImageNet normalized
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std


classifier.eval()
G_AB.eval()

all_gen_preds  = []
all_gen_labels = []
per_class_acc  = {}

for label_idx, cls in enumerate(cfg.CLASSES):
    sar_paths = sar_image_map[cls]
    random.shuffle(sar_paths)
    sar_paths = sar_paths[:cfg.INFER_PER_CLASS]

    sar_dataset = SARInferenceDataset(sar_paths, cfg.IMG_SIZE)
    sar_loader  = DataLoader(sar_dataset, batch_size=16,
                             shuffle=False, num_workers=0, pin_memory=True)

    class_preds = []
    with torch.no_grad():
        for sar_batch in sar_loader:
            sar_batch = sar_batch.to(device)

            # Step 1: SAR → Generated Optical
            with torch.cuda.amp.autocast(enabled=cfg.USE_AMP):
                gen_optical = G_AB(sar_batch)            # (B, 3, 256, 256) in [-1,1]

            # Step 2: Prepare for classifier (renormalize)
            clf_input = prepare_for_classifier(gen_optical)

            # Step 3: Classify
            with torch.cuda.amp.autocast(enabled=cfg.USE_AMP):
                logits = classifier(clf_input)
            preds = logits.argmax(dim=1).cpu().numpy()
            class_preds.extend(preds.tolist())

    true_labels = [label_idx] * len(class_preds)
    all_gen_preds.extend(class_preds)
    all_gen_labels.extend(true_labels)

    acc = sum(p == label_idx for p in class_preds) / len(class_preds)
    per_class_acc[cls] = acc
    print(f'  {cls:12s} → {acc*100:.1f}% ({sum(p==label_idx for p in class_preds)}/{len(class_preds)} correct)')

overall_gen_acc = sum(p == l for p, l in zip(all_gen_preds, all_gen_labels)) / len(all_gen_labels)
print(f'\nOverall Generated Accuracy: {overall_gen_acc*100:.2f}%')


In [ ]:
# ── CONFUSION MATRIX — GENERATED IMAGES ────────────────────────────────────
print(classification_report(all_gen_labels, all_gen_preds,
                             target_names=cfg.CLASSES, digits=4))

cm_gen = confusion_matrix(all_gen_labels, all_gen_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_gen, annot=True, fmt='d', cmap='Oranges',
            xticklabels=cfg.CLASSES, yticklabels=cfg.CLASSES, ax=ax)
ax.set_title(f'Generated Optical — Confusion Matrix (Acc: {overall_gen_acc*100:.1f}%)',
             fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'confusion_matrix_generated.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('Confusion matrix (generated) saved.')


In [ ]:
# ── 3-WAY COMPARISON TABLE ─────────────────────────────────────────────────
# Per-class accuracy from real val set
real_per_class = {}
for i, cls in enumerate(cfg.CLASSES):
    cls_labels = [l for l in real_labels if l == i]
    cls_preds  = [p for p, l in zip(real_preds, real_labels) if l == i]
    real_per_class[cls] = sum(p == i for p in cls_preds) / len(cls_preds) if cls_preds else 0.0

print('='*65)
print(f'{"Class":<14} {"Real Optical":>13} {"Generated":>11} {"Drop":>8}')
print('-'*65)
for cls in cfg.CLASSES:
    r = real_per_class[cls]
    g = per_class_acc[cls]
    d = g - r
    print(f'{cls:<14} {r*100:>12.1f}%  {g*100:>10.1f}%  {d*100:>+7.1f}%')
print('-'*65)
print(f'{"Overall":<14} {real_acc*100:>12.1f}%  {overall_gen_acc*100:>10.1f}%  {(overall_gen_acc-real_acc)*100:>+7.1f}%')
print('='*65)


In [ ]:
# ── COMPARISON BAR CHART ───────────────────────────────────────────────────
x     = np.arange(len(cfg.CLASSES))
width = 0.35

real_vals = [real_per_class[c]*100 for c in cfg.CLASSES]
gen_vals  = [per_class_acc[c]*100  for c in cfg.CLASSES]

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, real_vals, width, label='Real Optical',      color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, gen_vals,  width, label='Generated Optical', color='darkorange', alpha=0.85)

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

ax.axhline(y=real_acc*100,          color='steelblue',  linestyle='--', alpha=0.5, label=f'Real Overall ({real_acc*100:.1f}%)')
ax.axhline(y=overall_gen_acc*100,   color='darkorange', linestyle='--', alpha=0.5, label=f'Gen Overall ({overall_gen_acc*100:.1f}%)')

ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in cfg.CLASSES], fontsize=12)
ax.set_ylabel('Classification Accuracy (%)', fontsize=12)
ax.set_title('Real vs Generated Optical Image Classification Accuracy\n(ResNet-18, trained on real Sentinel-2 images)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'accuracy_comparison.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('Comparison bar chart saved.')


In [ ]:
# ── VISUAL SAMPLES: SAR → GENERATED → TRUE LABEL → PREDICTED LABEL ─────────
print('Generating visual sample grid...')

n_per_class  = 3
fig, axes = plt.subplots(len(cfg.CLASSES), n_per_class * 2,
                          figsize=(n_per_class * 2 * 3, len(cfg.CLASSES) * 3))

SAR_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

G_AB.eval()
classifier.eval()

for row, (label_idx, cls) in enumerate(enumerate(cfg.CLASSES)):
    sar_paths = sar_image_map[cls][:n_per_class]

    for col, sar_path in enumerate(sar_paths):
        # Load and display SAR
        img = Image.open(sar_path)
        arr = np.array(img)
        if arr.ndim == 3: arr = arr[:, :, 0]
        p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
        arr = np.clip(arr, p2, p98)
        if p98 > p2: arr = ((arr - p2) / (p98 - p2) * 255).astype(np.uint8)
        sar_disp = Image.fromarray(arr, mode='L')

        ax_sar = axes[row][col * 2]
        ax_sar.imshow(sar_disp, cmap='gray')
        ax_sar.set_title('SAR Input', fontsize=7)
        ax_sar.axis('off')

        # Generate optical
        sar_t = SAR_transform(sar_disp).unsqueeze(0).to(device)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=cfg.USE_AMP):
                gen_opt  = G_AB(sar_t)
                clf_in   = prepare_for_classifier(gen_opt)
                logit    = classifier(clf_in)
        pred_idx = logit.argmax(dim=1).item()
        pred_cls = cfg.CLASSES[pred_idx]
        correct  = pred_idx == label_idx

        # Denormalize generated image for display
        gen_disp = (gen_opt.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1) / 2
        gen_disp = gen_disp.clip(0, 1)

        ax_gen = axes[row][col * 2 + 1]
        ax_gen.imshow(gen_disp)
        color = 'green' if correct else 'red'
        ax_gen.set_title(f'Gen Optical\n→ {pred_cls}', fontsize=7, color=color)
        ax_gen.axis('off')

    # Row label
    axes[row][0].set_ylabel(cls.capitalize(), fontsize=10, fontweight='bold')

# Column headers
for col in range(n_per_class):
    axes[0][col*2].set_title(f'SAR #{col+1}', fontsize=8, color='gray')
    axes[0][col*2+1].set_title(f'Generated #{col+1}\n(pred)', fontsize=8, color='gray')

plt.suptitle('SAR → Generated Optical — Classification Results\n(Green=Correct, Red=Wrong)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'visual_samples.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('Visual samples grid saved.')


In [ ]:
# ── ZIP ALL OUTPUTS FOR DOWNLOAD ────────────────────────────────────────────
output_files = glob.glob(os.path.join(cfg.OUTPUT_DIR, '*.png'))
output_files.append(os.path.join(cfg.CHECKPOINT_DIR, 'best_classifier.pth'))

results_zip = os.path.join(cfg.WORKING_ROOT, 'classifier_results.zip')
with zipfile.ZipFile(results_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in output_files:
        if os.path.exists(f):
            zf.write(f, os.path.basename(f))
print(f'classifier_results.zip ({os.path.getsize(results_zip)/1e6:.1f} MB) → Output tab')

# ── FINAL SUMMARY ────────────────────────────────────────────────────────────
print()
print('='*65)
print('FINAL RESULTS SUMMARY')
print('='*65)
print(f'Classifier      : ResNet-18 pretrained (ImageNet) + fine-tuned')
print(f'Trained on      : Real Sentinel-2 optical images')
print(f'Training samples: {len(train_samples)}')
print()
print(f'{"Metric":<35} {"Value":>10}')
print('-'*50)
print(f'{"Real Optical Val Accuracy":<35} {real_acc*100:>9.2f}%')
print(f'{"Generated Optical Accuracy":<35} {overall_gen_acc*100:>9.2f}%')
print(f'{"Accuracy Drop (Real → Generated)":<35} {(overall_gen_acc-real_acc)*100:>+9.2f}%')
print()
print('Per-class breakdown:')
for cls in cfg.CLASSES:
    r = real_per_class[cls]
    g = per_class_acc[cls]
    print(f'  {cls:<12} Real: {r*100:.1f}%  Generated: {g*100:.1f}%  Drop: {(g-r)*100:+.1f}%')
print('='*65)
